In [ ]:
import requests
import pandas as pd
from tqdm import tqdm

# ---------------------------------------------------------
# 1. Läs API-nyckeln
# ---------------------------------------------------------
with open("credentials/Freshdesk_API-key.txt", "r") as f:
    API_KEY = f.read().strip()

DOMAIN = "intersolia.freshdesk.com"
BASE_URL = f"https://{DOMAIN}/api/v2"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

AUTH = (API_KEY, "X")

# ---------------------------------------------------------
# 2. Funktion: hämta alla tickets med pagination
# ---------------------------------------------------------
def fetch_all_tickets(per_page=100):
    all_tickets = []
    page = 1

    while True:
        url = f"{BASE_URL}/tickets?per_page={per_page}&page={page}"
        resp = requests.get(url, headers=HEADERS, auth=AUTH)

        # Hantera fel enligt dokumentationen
        if resp.status_code != 200:
            print(f"Fel vid hämtning av tickets (page {page}): {resp.status_code}")
            print(resp.text)
            break

        data = resp.json()
        if not data:
            break

        all_tickets.extend(data)
        page += 1

    return all_tickets

# ---------------------------------------------------------
# 3. Funktion: hämta audits för en ticket
# ---------------------------------------------------------
def fetch_ticket_audits(ticket_id):
    url = f"{BASE_URL}/tickets/{ticket_id}/audits"
    resp = requests.get(url, headers=HEADERS, auth=AUTH)

    if resp.status_code != 200:
        print(f"Fel vid hämtning av audits för ticket {ticket_id}: {resp.status_code}")
        print(resp.text)
        return []

    return resp.json()

# ---------------------------------------------------------
# 4. Hämta tickets
# ---------------------------------------------------------
print("Hämtar tickets...")
tickets = fetch_all_tickets()
df_tickets = pd.DataFrame(tickets)

print("\nTicketfält:")
print(df_tickets.columns.tolist())

# ---------------------------------------------------------
# 5. Exportera headers + 10 rader till Excel
# ---------------------------------------------------------
df_tickets.head(10).to_excel("tickets_sample.xlsx", index=False)
print("Exporterat: tickets_sample.xlsx")

# ---------------------------------------------------------
# 6. Hämta audits
# ---------------------------------------------------------
audit_rows = []

print("\nHämtar audits...")
for _, row in tqdm(df_tickets.iterrows(), total=len(df_tickets)):
    ticket_id = row["id"]
    audits = fetch_ticket_audits(ticket_id)

    for a in audits:
        for event in a.get("events", []):
            audit_rows.append({
                "ticket_id": ticket_id,
                "audit_time": a.get("created_at"),
                "user_id": a.get("user_id"),
                "field": event.get("field_name"),
                "value": event.get("value")
            })

df_audits = pd.DataFrame(audit_rows)

df_audits.head(10).to_excel("audits_sample.xlsx", index=False)
print("Exporterat: audits_sample.xlsx")


In [ ]:
df_tickets.head(10).to_excel("tickets_sample.xlsx", index=False)


In [10]:
import requests
import json
import sys

# Läs API-nyckel från fil
with open("credentials/Freshdesk_API-key.txt", "r") as f:
    API_KEY = f.read().strip()

domain = "intersolia.freshdesk.com"
url = f"https://{domain}/api/v2/ticket_fields"

# Anropa Freshdesk
resp = requests.get(url, auth=(API_KEY, "X"), timeout=30)

print("HTTP status:", resp.status_code)

# Försök tolka JSON, annars skriv rå text
try:
    payload = resp.json()
except ValueError:
    payload = None

if resp.status_code == 200 and payload is not None:
    print(json.dumps(payload, indent=2, ensure_ascii=False))
else:
    # Visa användbar felsökningsinfo
    print("Response headers:")
    for k, v in resp.headers.items():
        print(f"{k}: {v}")
    if payload:
        print("\nJSON body:")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
    else:
        print("\nBody (raw):")
        print(resp.text)

    # Extra hint baserat på vanliga svar
    if resp.status_code in (401, 403):
        print("\nFel: Ogiltiga eller otillräckliga rättigheter för denna endpoint.")
        print("Kontrollera att API-nyckeln tillhör en användare med rätt att läsa ticket_fields (admin/Manage Ticket Fields).")
    elif resp.status_code == 404:
        print("\nFel: Endpoint hittades inte. Kontrollera domän och URL.")
    sys.exit(1)


HTTP status: 200
[
  {
    "id": 14000273569,
    "name": "requester",
    "label": "Search a requester",
    "description": "Ticket requester",
    "position": 1,
    "required_for_closure": false,
    "required_for_agents": true,
    "type": "default_requester",
    "default": true,
    "customers_can_edit": true,
    "customers_can_filter": false,
    "label_for_customers": "Requester",
    "required_for_customers": true,
    "displayed_to_customers": true,
    "created_at": "2016-04-15T12:00:05Z",
    "updated_at": "2026-03-26T07:34:28Z",
    "portal_cc": false,
    "portal_cc_to": "company"
  },
  {
    "id": 14000335469,
    "name": "produkt",
    "label": "Type of service",
    "description": "",
    "position": 2,
    "required_for_closure": false,
    "required_for_agents": false,
    "type": "custom_dropdown",
    "default": false,
    "customers_can_edit": true,
    "customers_can_filter": false,
    "label_for_customers": "Type of service",
    "required_for_customers": tru

In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import requests
import json
import csv
from pathlib import Path
import traceback
from datetime import datetime, timezone

# Konfiguration
CREDENTIALS_PATH = Path("credentials/Freshdesk_API-key.txt")
DOMAIN = "intersolia.freshdesk.com"
OUT_DIR = Path("data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSON_PATH = OUT_DIR / "ticket_fields.json"
CSV_PATH = OUT_DIR / "freshdesk_status_lookup.csv"
META_OUT = OUT_DIR / "status_field_extracted.json"
TIMEOUT = 30

def main():
    # Läs API-nyckel
    if not CREDENTIALS_PATH.exists():
        raise FileNotFoundError(f"Credentials file not found: {CREDENTIALS_PATH}")

    with CREDENTIALS_PATH.open("r", encoding="utf-8") as f:
        API_KEY = f.read().strip()

    if not API_KEY:
        raise ValueError("API key is empty")

    # Hämta ticket_fields från Freshdesk
    url = f"https://{DOMAIN}/api/v2/ticket_fields"
    try:
        resp = requests.get(url, auth=(API_KEY, "X"), timeout=TIMEOUT)
    except requests.RequestException as e:
        raise RuntimeError(f"Request failed: {e}") from e

    print(f"HTTP status: {resp.status_code}")

    # Spara rå JSON oavsett status (för felsökning)
    try:
        RAW_JSON_PATH.write_text(resp.text, encoding="utf-8")
        print(f"Saved raw response to {RAW_JSON_PATH}")
    except Exception as e:
        print(f"WARNING: Could not save raw JSON: {e}")

    # Försök tolka JSON
    try:
        payload = resp.json()
    except ValueError:
        raise RuntimeError("Response is not valid JSON:\n" + resp.text)

    # Hitta statusfältet
    status_field = None
    for field in payload:
        if field.get("name") == "status":
            status_field = field
            break

    if not status_field:
        names = [f.get("name") for f in payload if "name" in f]
        msg = "Could not find a field with name == 'status' in ticket_fields JSON.\nAvailable field names:\n"
        msg += "\n".join(f" - {n}" for n in names)
        raise RuntimeError(msg)

    # Extrahera choices
    choices = status_field.get("choices", {})
    if not isinstance(choices, dict) or not choices:
        raise RuntimeError("'choices' for status field is empty or not a dict:\n" + json.dumps(status_field, indent=2, ensure_ascii=False))

    # Skriv CSV med status_id,status_name
    extracted_at = datetime.now(timezone.utc).isoformat()
    try:
        with CSV_PATH.open("w", newline="", encoding="utf-8") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["status_id", "status_name", "extracted_at_utc"])
            # Sortera efter numerisk nyckel om möjligt, annars lexikografiskt
            def sort_key(k):
                try:
                    return int(k)
                except Exception:
                    return k
            for sid in sorted(choices.keys(), key=sort_key):
                sname = choices[sid]
                writer.writerow([sid, sname, extracted_at])
        print(f"Saved status lookup CSV to {CSV_PATH}")
    except Exception as e:
        raise RuntimeError(f"Could not write CSV: {e}") from e

    # Spara status_field som JSON (inkl. metadata)
    try:
        meta = {
            "fetched_at_utc": extracted_at,
            "domain": DOMAIN,
            "status_field": status_field
        }
        META_OUT.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"Saved extracted status field JSON to {META_OUT}")
    except Exception as e:
        print(f"WARNING: Could not save extracted JSON: {e}")

    # Returnera paths för vidare användning i Notebook
    return {
        "raw_json": str(RAW_JSON_PATH),
        "csv": str(CSV_PATH),
        "meta_json": str(META_OUT),
        "http_status": resp.status_code
    }

if __name__ == "__main__":
    try:
        result = main()
        print("Done. Result:", result)
    except Exception as exc:
        print("ERROR:", str(exc))
        traceback.print_exc()


HTTP status: 200
Saved raw response to data\ticket_fields.json
Saved status lookup CSV to data\freshdesk_status_lookup.csv
Saved extracted status field JSON to data\status_field_extracted.json
Done. Result: {'raw_json': 'data\\ticket_fields.json', 'csv': 'data\\freshdesk_status_lookup.csv', 'meta_json': 'data\\status_field_extracted.json', 'http_status': 200}


In [17]:
# Jupyter cell: linear_introspect_and_snapshot.ipynb
import requests, json, time, traceback
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

# Konfiguration
CREDENTIALS_PATH = Path("credentials/Linear_API-key.txt")
OUT_DIR = Path("data/linear")
OUT_DIR.mkdir(parents=True, exist_ok=True)
API_URL = "https://api.linear.app/graphql"
PAGE_SIZE = 100
TIMEOUT = 60
SLEEP_BETWEEN_PAGES = 0.2
SAMPLE_ROWS = 50

# Läs API-nyckel (Linear kräver nyckeln ensam i Authorization)
if not CREDENTIALS_PATH.exists():
    raise FileNotFoundError(f"Missing Linear API key file: {CREDENTIALS_PATH}")
API_KEY = CREDENTIALS_PATH.read_text(encoding="utf-8").strip()
if not API_KEY:
    raise ValueError("Linear API key is empty")

HEADERS = {
    "Authorization": API_KEY,
    "Content-Type": "application/json",
    "Accept": "application/json",
    "User-Agent": "linear-jupyter-introspect-snapshot/1.0"
}

def graphql_post_raw(payload):
    resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=TIMEOUT)
    try:
        resp.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"HTTP error: {e}\nStatus: {resp.status_code}\nBody: {resp.text}") from e
    data = resp.json()
    if "errors" in data:
        raise RuntimeError(f"GraphQL errors: {json.dumps(data['errors'], indent=2, ensure_ascii=False)}")
    return data

# 1) Introspektion: hämta schema och identifiera Issue-typen och dess fält
print("Running GraphQL introspection...")
introspection_query = {
    "query": """
    query IntrospectionQuery {
      __schema {
        queryType { name }
        types {
          name
          kind
          fields {
            name
            type { name kind ofType { name kind } }
            args { name }
          }
        }
      }
    }
    """
}
introspect = graphql_post_raw(introspection_query)
schema = introspect.get("data", {}).get("__schema", {})
introspect_path = OUT_DIR / "introspection_schema.json"
introspect_path.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved introspection schema to {introspect_path}")

# Hitta root query fields
root_query_name = schema.get("queryType", {}).get("name")
root_fields = []
for t in schema.get("types", []):
    if t.get("name") == root_query_name and t.get("fields"):
        root_fields = [f.get("name") for f in t.get("fields")]
        break

print("\nRoot query fields (sample):")
print(", ".join(root_fields[:80]))

# Hitta Issue-typen (sök efter typ med fälten identifier/title/createdAt)
issue_type = None
for t in schema.get("types", []):
    if t.get("name") == "Issue":
        issue_type = t
        break
if not issue_type:
    # fallback: hitta typ som innehåller identifier/title/createdAt
    for t in schema.get("types", []):
        fnames = {f.get("name") for f in (t.get("fields") or [])}
        if {"title", "createdAt"}.issubset(fnames):
            issue_type = t
            break

if not issue_type:
    raise RuntimeError("Could not detect Issue type from introspection. Inspect introspection_schema.json manually.")

issue_fields = [f.get("name") for f in (issue_type.get("fields") or [])]
print(f"\nDetected Issue type: {issue_type.get('name')}")
print("Available Issue fields (sample):")
print(", ".join(issue_fields[:80]))

# 2) Bestäm vilka fält vi säkert kan begära (whitelist ∩ schema)
recommended = [
    "id", "number", "identifier", "title", "description", "createdAt", "updatedAt",
    "archivedAt", "archived", "priority", "estimate", "trashed",
    "state", "assignee", "project", "team", "labels"
]
available = [f for f in recommended if f in issue_fields]

# Prefer 'number' if present, else 'identifier' if present
if "number" in available and "identifier" in available:
    # keep both
    pass
elif "number" in available and "identifier" not in available:
    # ensure 'number' used
    if "identifier" in available:
        available.remove("identifier")
elif "identifier" in available and "number" not in available:
    pass

if not available:
    raise RuntimeError("No recommended Issue fields found in schema. Inspect introspection_schema.json.")

print("\nWill request these Issue fields:", available)

# 3) Bygg dynamisk GraphQL query utan orderBy (för att undvika enum-valideringsfel)
# Hantera nested selections för state, assignee, project, team, labels
nested = {
    "state": "{ id name type }",
    "assignee": "{ id name email }",
    "project": "{ id name }",
    "team": "{ id name }",
    "labels": "{ nodes { id name } }"
}
selections = []
for f in available:
    if f in nested:
        selections.append(f"{f} {nested[f]}")
    else:
        selections.append(f)
sel_block = "\n              ".join(selections)

issues_query = f"""
query($first:Int!, $after:String) {{
  issues(first:$first, after:$after) {{
    nodes {{
              {sel_block}
    }}
    pageInfo {{ hasNextPage endCursor }}
  }}
}}
"""

# 4) Hämta alla issues paginerat (utan orderBy)
print("\nFetching issues (paginated) without orderBy)...")
issues = []
cursor = None
page = 0
try:
    while True:
        page += 1
        print(f"Fetching page {page} (after cursor: {cursor}) ...")
        payload = {"query": issues_query, "variables": {"first": PAGE_SIZE, "after": cursor}}
        resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=TIMEOUT)
        try:
            resp.raise_for_status()
        except Exception as e:
            raise RuntimeError(f"HTTP error: {e}\nStatus: {resp.status_code}\nBody: {resp.text}") from e
        data = resp.json()
        if "errors" in data:
            raise RuntimeError(f"GraphQL errors: {json.dumps(data['errors'], indent=2, ensure_ascii=False)}")
        block = data["data"]["issues"]["nodes"]
        issues.extend(block)
        page_info = data["data"]["issues"]["pageInfo"]
        if page_info.get("hasNextPage"):
            cursor = page_info.get("endCursor")
            time.sleep(SLEEP_BETWEEN_PAGES)
        else:
            break
    print(f"Fetched total issues: {len(issues)}")
except Exception as e:
    tb = traceback.format_exc()
    print("Error while fetching issues:", e)
    print(tb)
    err_path = OUT_DIR / f"issues_fetch_error_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
    err_path.write_text(json.dumps({"error": str(e), "traceback": tb}, indent=2, ensure_ascii=False), encoding="utf-8")
    if issues:
        partial_path = OUT_DIR / f"issues_partial_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
        partial_path.write_text(json.dumps(issues, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"Saved partial issues to {partial_path}")
    raise

# 5) Normalisera issues till DataFrame (anpassa efter tillgängliga fält)
def normalize_issue(it):
    row = {"snapshot_ts": datetime.now(timezone.utc).isoformat()}
    # scalar fields
    for f in available:
        if f in {"state","assignee","project","team"}:
            val = it.get(f) or {}
            row[f + "_id"] = val.get("id")
            row[f + "_name"] = val.get("name")
            if f == "assignee":
                row["assignee_email"] = val.get("email")
            if f == "state":
                row["state_type"] = val.get("type")
        elif f == "labels":
            labels = ";".join([lbl.get("name","") for lbl in (it.get("labels") or {}).get("nodes", [])])
            row["labels"] = labels
        else:
            row[f] = it.get(f)
    return row

rows = [normalize_issue(it) for it in issues]
df = pd.DataFrame(rows)

# 6) Spara rå JSON och Excel med flera flikar
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw_issues_path = OUT_DIR / f"issues_raw_{ts}.json"
raw_introspect_path = OUT_DIR / f"introspection_schema_{ts}.json"
excel_path = OUT_DIR / f"linear_snapshot_{ts}.xlsx"

raw_issues_path.write_text(json.dumps(issues, indent=2, ensure_ascii=False), encoding="utf-8")
Path(raw_introspect_path).write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved raw issues JSON to {raw_issues_path}")
print(f"Saved introspection JSON to {raw_introspect_path}")

df_sample = df.head(SAMPLE_ROWS).copy()
types_issue_fields_df = pd.DataFrame({"issue_field": issue_fields})
root_fields_df = pd.DataFrame({"root_query_field": root_fields})

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="issues_snapshot", index=False)
    df_sample.to_excel(writer, sheet_name="sample_issues", index=False)
    types_issue_fields_df.to_excel(writer, sheet_name="types_issue_fields", index=False)
    root_fields_df.to_excel(writer, sheet_name="introspection_root_fields", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="errors", index=False)

print(f"\nExcel workbook created: {excel_path}")
print("Done. Sheets: issues_snapshot, sample_issues, types_issue_fields, introspection_root_fields, errors")

# Visa preview i notebook
if not df.empty:
    display(df.head(min(10, len(df))))
else:
    print("No issues returned.")


Running GraphQL introspection...
Saved introspection schema to data\linear\introspection_schema.json

Root query fields (sample):
agentActivities, agentActivity, agentSessions, agentSession, agentSessionSandbox, applicationInfo, attachments, attachment, attachmentsForURL, attachmentSources, auditEntryTypes, auditEntries, availableUsers, authenticationSessions, ssoUrlFromEmail, comments, comment, projects, project, projectFilterSuggestion, customViews, customView, customViewDetailsSuggestion, customViewHasSubscribers, customerNeeds, customerNeed, issueTitleSuggestionFromCustomerRequest, customers, customer, customerStatuses, customerStatus, customerTiers, customerTier, cycles, cycle, documentContentHistory, documentContentHistoryEntries, documents, document, emailIntakeAddress, emojis, emoji, entityExternalLink, externalUsers, externalUser, favorites, favorite, fetchData, initiativeRelations, initiativeRelation, initiatives, initiative, initiativeToProjects, initiativeToProject, initiat

,snapshot_ts,id,number,identifier,title,description,createdAt,updatedAt,archivedAt,priority,...,state_name,state_type,assignee_id,assignee_name,assignee_email,project_id,project_name,team_id,team_name,labels
0,2026-05-29T11:31:54.208858+00:00,f269163b-5255-415f-b052-5885298e4387,868,OPEX-868,Execution of CPU_Job (iChemSqlLobotNE),Status: Critical\n\nMessage:\n\nEntityId: 3196...,2026-05-29T11:01:43.020Z,2026-05-29T11:06:14.017Z,None,0,...,In Progress,started,56304f91-ec36-44ec-b53e-056ed5f898ab,John Christoffersson,john.christoffersson@intersolia.com,NaN,NaN,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,OwlMonitor
1,2026-05-29T11:31:54.208912+00:00,cde58e61-ec55-4cd3-8525-227e1f9cc1e1,867,OPEX-867,FD322168 - Flytt av SKF GmbH - 2026-06-02,[https://customer.intersolia.com/a/tickets/322...,2026-05-29T07:46:24.633Z,2026-05-29T08:08:25.921Z,None,3,...,Todo - Fast,unstarted,d386ff68-088f-4e7d-89c7-1fef50e18e12,Kenny Lindblom,kenny.lindblom@intersolia.com,e9a4edcc-c562-4b25-97c0-a64beb40441b,iChemistry,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,Customer Move;Routine
2,2026-05-29T11:31:54.208936+00:00,79158576-c506-4617-936c-07cd3a84bea1,866,OPEX-866,FD320474 - Fel i Finsk template - iPub,[https://customer.intersolia.com/a/tickets/320...,2026-05-29T07:38:56.632Z,2026-05-29T08:06:53.584Z,None,2,...,Todo - Bugs etc,unstarted,NaN,NaN,NaN,03999df5-8fe3-4644-b23c-f7f603ce741a,iPublisher,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,Bug
3,2026-05-29T11:31:54.208949+00:00,a105182e-0e17-4b5f-a43b-e1c4ce1e270e,865,OPEX-865,FD323525 - Open Access funkar inte,[https://customer.intersolia.com/a/tickets/323...,2026-05-29T07:28:50.765Z,2026-05-29T08:55:47.413Z,None,1,...,Done,completed,38694ffb-5192-49ec-889b-2611a2cb4d24,Jaakko Tarhanen,jaakko.tarhanen@intersolia.com,e9a4edcc-c562-4b25-97c0-a64beb40441b,iChemistry,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,Bug
4,2026-05-29T11:31:54.208961+00:00,3dfb4194-3ee3-4e71-b99a-3a43e26f3962,864,OPEX-864,FD322763 - Osignerade RA visas i appen,[https://customer.intersolia.com/a/tickets/322...,2026-05-29T07:24:57.776Z,2026-05-29T08:06:16.662Z,None,2,...,Todo - Bugs etc,unstarted,NaN,NaN,NaN,NaN,NaN,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,Bug;Investigation;Mobil App
5,2026-05-29T11:31:54.208977+00:00,3b328775-fed3-45dc-b0ca-2b95a5d63f20,863,OPEX-863,Execution of CPU_Job (iChemSqlLobotNE),Status: Critical\n\nMessage:\n\nEntityId: 3196...,2026-05-29T07:22:12.712Z,2026-05-29T11:01:23.714Z,None,0,...,Done,completed,56304f91-ec36-44ec-b53e-056ed5f898ab,John Christoffersson,john.christoffersson@intersolia.com,NaN,NaN,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,OwlMonitor
6,2026-05-29T11:31:54.208987+00:00,c26391d0-766f-4a3c-8eb3-4aeb71558f62,862,OPEX-862,FD322830 Chemsoft - SDS Link in Risk assessmen...,[https://support.chemsoft.eu/a/tickets/322830]...,2026-05-29T07:18:37.695Z,2026-05-29T09:37:34.019Z,None,3,...,In Progress,started,8e68beac-3460-431b-967d-a038c1667bf3,Anton Larsson,anton.larsson@intersolia.com,89dfac53-a8c0-46e4-9620-917ce1dfb50c,Chemsoft,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,Bug
7,2026-05-29T11:31:54.208996+00:00,29001d28-b5c3-4f89-8cc6-4f5c3a725445,861,OPEX-861,FD 322071 - Ändra toppavdelningsens namn för B...,[https://customer.intersolia.com/a/tickets/322...,2026-05-29T07:13:00.510Z,2026-05-29T08:46:29.480Z,None,3,...,Done,completed,8e68beac-3460-431b-967d-a038c1667bf3,Anton Larsson,anton.larsson@intersolia.com,NaN,NaN,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,
8,2026-05-29T11:31:54.209007+00:00,afc7cf29-e609-491f-80ca-3165ac03e8d7,860,OPEX-860,FD323415 - Sätt upp exponeringsregister - Nabo...,[https://customer.intersolia.com/a/tickets/323...,2026-05-29T07:08:06.196Z,2026-05-29T07:20:49.135Z,None,3,...,Done,completed,8e68beac-3460-431b-967d-a038c1667bf3,Anton Larsson,anton.larsson@intersolia.com,NaN,NaN,f355d2e6-ba8a-4dee-bb03-cbf3095e8d05,Operational,
9,2026-05-29T11:31:54.209018+00:00,ac89b690-cc8a-4186-a7e6-0d025fda2f74,859,OPEX-859,FD 323417 -

EDA - complete 2026-01-01 to 2026-05-31


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Backfill weekly raw files + append CSV + EDA for Freshdesk and Linear
Date range (inclusive): 2025-06-01 -> 2026-05-31
- Creates one raw JSON file per week in raw/<system>/
- Appends normalized rows to master CSVs in data/<system>/
- Produces EDA CSVs for the full range
Requirements: requests, pandas, python-dateutil
Run: python backfill_weekly.py
"""

import requests
import json
import time
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dateutil import parser as dateparse
import pandas as pd
import csv
import traceback
from collections import Counter

# -------------------------
# CONFIG
# -------------------------
PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "raw"
DATA_DIR = PROJECT_ROOT / "data"
LOGS_DIR = PROJECT_ROOT / "logs"
CREDENTIALS_DIR = PROJECT_ROOT / "credentials"

LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"
LINEAR_API_URL = "https://api.linear.app/graphql"
LINEAR_PAGE_SIZE = 100

FRESHDESK_API_KEY_PATH = CREDENTIALS_DIR / "Freshdesk_API-key.txt"
FRESHDESK_DOMAIN = "intersolia.freshdesk.com"
FRESHDESK_TICKETS_ENDPOINT = f"https://{FRESHDESK_DOMAIN}/api/v2/tickets"
FRESHDESK_PAGE_SIZE = 100

# Date range (inclusive)
START_DATE = datetime(2025, 6, 1, 0, 0, 0, tzinfo=timezone.utc)
END_DATE = datetime(2026, 5, 31, 23, 59, 59, tzinfo=timezone.utc)

# Ensure directories
for p in [RAW_DIR, DATA_DIR, LOGS_DIR, RAW_DIR / "freshdesk", RAW_DIR / "linear", DATA_DIR / "freshdesk", DATA_DIR / "linear"]:
    p.mkdir(parents=True, exist_ok=True)

# Timestamp helper
def utc_now_ts():
    return datetime.now(timezone.utc)

def ts_for_filename(dt=None):
    if dt is None:
        dt = utc_now_ts()
    return dt.strftime("%Y%m%dT%H%M%SZ")

# Atomic write helpers
def atomic_write_text(path: Path, text: str, encoding="utf-8"):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding=encoding)
    tmp.replace(path)

def atomic_write_bytes(path: Path, b: bytes):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(b)
    tmp.replace(path)

# Append CSV atomically (append mode)
def append_csv_atomic(target: Path, df: pd.DataFrame):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(".tmp")
    if target.exists():
        # write without header then append bytes
        df.to_csv(tmp, index=False, header=False)
        with open(target, "ab") as f_target, open(tmp, "rb") as f_tmp:
            f_target.write(f_tmp.read())
        tmp.unlink(missing_ok=True)
    else:
        df.to_csv(tmp, index=False, header=True)
        tmp.replace(target)

# Read API key
def read_api_key(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing API key file: {path}")
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API key file {path} is empty")
    return key

# Retry wrappers
def http_post_with_retry(url, headers=None, json_payload=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.post(url, headers=headers, json=json_payload, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

def http_get_with_retry(url, auth=None, params=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.get(url, auth=auth, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

# -------------------------
# Field lists (exact per user)
# -------------------------
FRESHDESK_FIELDS = [
    'cc_emails','fwd_emails','reply_cc_emails','ticket_cc_emails','ticket_bcc_emails',
    'fr_escalated','spam','email_config_id','group_id','priority','requester_id',
    'responder_id','source','company_id','status','subject','association_type',
    'support_email','to_emails','product_id','id','type','due_by','fr_due_by',
    'is_escalated','custom_fields','created_at','updated_at','associated_tickets_count',
    'tags','source_info','structured_description','internal_agent_id','internal_group_id',
    'sentiment_score','initial_sentiment_score','nr_due_by','nr_escalated','form_id'
]

LINEAR_FIELDS = [
    "id","number","identifier","title","description","createdAt","updatedAt","archivedAt",
    "priority","estimate","trashed","state_id","state_name","state_type",
    "assignee_id","assignee_name","assignee_email","project_id","project_name",
    "team_id","team_name","labels"
]

# Suggested handling mapping (used in EDA output)
SUGGESTED_HANDLING_FD = {
    "cc_emails":"flatten to ';' string; keep raw JSON",
    "fwd_emails":"flatten to ';' string; keep raw JSON",
    "reply_cc_emails":"flatten to ';' string; keep raw JSON",
    "ticket_cc_emails":"flatten to ';' string; keep raw JSON",
    "ticket_bcc_emails":"flatten to ';' string; keep raw JSON",
    "fr_escalated":"normalize to boolean",
    "spam":"normalize to boolean",
    "email_config_id":"keep as string; nullable",
    "group_id":"keep as string; map to DimGroup if needed",
    "priority":"normalize to numeric or enum; document mapping",
    "requester_id":"keep as string; link to DimPerson if available",
    "responder_id":"keep as string; link to DimPerson if available",
    "source":"keep as string; canonicalize values",
    "company_id":"keep as string; map to DimCompany if needed",
    "status":"normalize to numeric + map to status_name in silver",
    "subject":"trim whitespace; keep raw",
    "association_type":"keep as string; nullable",
    "support_email":"validate email format; keep as string",
    "to_emails":"flatten to ';' string",
    "product_id":"keep as string; map to DimProduct if used",
    "id":"primary id for Freshdesk rows; keep as string",
    "type":"keep as string; map to canonical types if needed",
    "due_by":"normalize to UTC ISO; nullable",
    "fr_due_by":"normalize to UTC ISO; nullable",
    "is_escalated":"normalize to boolean",
    "custom_fields":"keep raw JSON; optionally extract known keys",
    "created_at":"normalize to UTC ISO; index for time joins",
    "updated_at":"normalize to UTC ISO; index for incremental fetch",
    "associated_tickets_count":"keep numeric; nullable",
    "tags":"flatten to ';' string; note cardinality",
    "source_info":"keep raw JSON; extract fields if needed",
    "structured_description":"keep raw JSON; consider text extraction",
    "internal_agent_id":"keep as string; map to DimPerson if used",
    "internal_group_id":"keep as string; map to DimGroup if used",
    "sentiment_score":"keep numeric; normalize scale if needed",
    "initial_sentiment_score":"keep numeric; normalize scale if needed",
    "nr_due_by":"normalize to UTC ISO; nullable",
    "nr_escalated":"normalize to boolean",
    "form_id":"keep as string; nullable"
}

SUGGESTED_HANDLING_LN = {
    "id":"keep as string; PK",
    "number":"keep numeric",
    "identifier":"keep as string; may be null",
    "title":"trim whitespace; keep raw",
    "description":"keep raw; consider storing large text in raw JSON only",
    "createdAt":"normalize to UTC ISO; index for time joins",
    "updatedAt":"normalize to UTC ISO; index for incremental fetch",
    "archivedAt":"normalize to UTC ISO; nullable",
    "priority":"normalize to numeric or enum; document mapping",
    "estimate":"keep numeric; document unit",
    "trashed":"normalize to boolean",
    "state_id":"keep as string; map to DimState",
    "state_name":"keep as string",
    "state_type":"keep as string",
    "assignee_id":"keep as string; map to DimPerson",
    "assignee_name":"keep as string",
    "assignee_email":"validate email format",
    "project_id":"keep as string; map to DimProject",
    "project_name":"keep as string",
    "team_id":"keep as string; map to DimTeam",
    "team_name":"keep as string",
    "labels":"flatten to ';' string; consider label table in silver"
}

# -------------------------
# Normalizers (match user's notebook)
# -------------------------
def normalize_freshdesk_ticket(t, snapshot_ts_iso, raw_path):
    return {
        "snapshot_ts": snapshot_ts_iso,
        "ticket_id": t.get("id"),
        "subject": t.get("subject"),
        "description": t.get("description"),
        "created_at": t.get("created_at"),
        "updated_at": t.get("updated_at"),
        "status_id": t.get("status"),
        "status_name": None,
        "priority": t.get("priority"),
        "requester_id": t.get("requester_id"),
        "requester_name": None,
        "assignee_id": t.get("responder_id"),
        "assignee_name": None,
        "tags": ";".join(t.get("tags", [])) if t.get("tags") else "",
        "raw_json_path": str(raw_path)
    }

def normalize_linear_issue(node, snapshot_ts_iso, raw_path):
    labels_nodes = (node.get("labels") or {}).get("nodes", [])
    labels = ";".join([lbl.get("name","") for lbl in labels_nodes])
    state = node.get("state") or {}
    assignee = node.get("assignee") or {}
    project = node.get("project") or {}
    team = node.get("team") or {}
    return {
        "snapshot_ts": snapshot_ts_iso,
        "id": node.get("id"),
        "number": node.get("number"),
        "identifier": node.get("identifier"),
        "title": node.get("title"),
        "description": node.get("description"),
        "createdAt": node.get("createdAt"),
        "updatedAt": node.get("updatedAt"),
        "archivedAt": node.get("archivedAt"),
        "priority": node.get("priority"),
        "estimate": node.get("estimate"),
        "trashed": node.get("trashed"),
        "state_id": state.get("id"),
        "state_name": state.get("name"),
        "state_type": state.get("type"),
        "assignee_id": assignee.get("id"),
        "assignee_name": assignee.get("name"),
        "assignee_email": assignee.get("email"),
        "project_id": project.get("id"),
        "project_name": project.get("name"),
        "team_id": team.get("id"),
        "team_name": team.get("name"),
        "labels": labels,
        "raw_json_path": str(raw_path)
    }

# -------------------------
# Week windows generator
# -------------------------
def week_windows(start_dt: datetime, end_dt: datetime):
    cur = start_dt
    # align to week start (Monday) for nicer filenames, but include start_dt exactly
    # We'll yield windows [w_start, w_end] inclusive
    while cur <= end_dt:
        w_start = cur
        w_end = min(cur + timedelta(days=6, hours=23, minutes=59, seconds=59), end_dt)
        yield (w_start, w_end)
        cur = w_end + timedelta(seconds=1)

# -------------------------
# Fetch Freshdesk tickets (paginated) and filter by date range
# - Handles HTTP 400 by stopping paging for that window (logged)
# -------------------------
def fetch_freshdesk_tickets(api_key: str, start_dt: datetime, end_dt: datetime):
    auth = (api_key, "X")
    tickets = []
    page = 1
    params = {"per_page": FRESHDESK_PAGE_SIZE, "page": page}
    params["updated_since"] = start_dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")
    errors = []
    while True:
        params["page"] = page
        try:
            resp = http_get_with_retry(FRESHDESK_TICKETS_ENDPOINT, auth=auth, params=params, max_retries=3, backoff=1.0, timeout=60)
            block = resp.json()
        except requests.exceptions.HTTPError as he:
            status = None
            try:
                status = he.response.status_code
            except Exception:
                status = None
            if status == 400:
                errors.append({"page": page, "error": f"HTTP 400 Bad Request (stop paging): {he}"})
                break
            else:
                errors.append({"page": page, "error": str(he), "traceback": traceback.format_exc()})
                break
        except Exception as e:
            errors.append({"page": page, "error": str(e), "traceback": traceback.format_exc()})
            break
        if not isinstance(block, list):
            errors.append({"page": page, "error": "Unexpected response type", "body": block})
            break
        tickets.extend(block)
        if len(block) < FRESHDESK_PAGE_SIZE:
            break
        page += 1
        # safety: avoid infinite loops
        if page > 500:
            errors.append({"page": page, "error": "Exceeded page safety limit (500). Stopping."})
            break
        time.sleep(0.2)
    # Local filter by created_at or updated_at within [start_dt, end_dt]
    filtered = []
    for t in tickets:
        created = None
        updated = None
        try:
            created = dateparse.parse(t.get("created_at")) if t.get("created_at") else None
        except Exception:
            created = None
        try:
            updated = dateparse.parse(t.get("updated_at")) if t.get("updated_at") else None
        except Exception:
            updated = None
        if (created and start_dt <= created <= end_dt) or (updated and start_dt <= updated <= end_dt):
            filtered.append(t)
    return filtered, errors

# -------------------------
# Fetch Linear issues (GraphQL paginated) and filter by date range
# - Uses selection matching user's flattened fields
# -------------------------
def fetch_linear_issues(api_key: str, start_dt: datetime, end_dt: datetime):
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "linear-bronze-snapshot/1.0"
    }
    sel = """
        id
        number
        identifier
        title
        description
        createdAt
        updatedAt
        archivedAt
        priority
        estimate
        trashed
        state { id name type }
        assignee { id name email }
        project { id name }
        team { id name }
        labels(first:50) { nodes { id name } }
    """
    query = f"""
    query($first:Int!, $after:String) {{
      issues(first:$first, after:$after) {{
        nodes {{
          {sel}
        }}
        pageInfo {{ hasNextPage endCursor }}
      }}
    }}
    """
    issues = []
    cursor = None
    page = 0
    errors = []
    while True:
        page += 1
        payload = {"query": query, "variables": {"first": LINEAR_PAGE_SIZE, "after": cursor}}
        try:
            resp = http_post_with_retry(LINEAR_API_URL, headers=headers, json_payload=payload, max_retries=3, backoff=1.0, timeout=60)
            data = resp.json()
        except Exception as e:
            errors.append({"page": page, "error": str(e), "traceback": traceback.format_exc()})
            break
        if "errors" in data:
            errors.append({"page": page, "graphql_errors": data["errors"]})
            break
        block = data["data"]["issues"]["nodes"]
        # local filter by createdAt or updatedAt
        for it in block:
            created = None
            updated = None
            try:
                created = dateparse.parse(it.get("createdAt")) if it.get("createdAt") else None
            except Exception:
                created = None
            try:
                updated = dateparse.parse(it.get("updatedAt")) if it.get("updatedAt") else None
            except Exception:
                updated = None
            if (created and start_dt <= created <= end_dt) or (updated and start_dt <= updated <= end_dt):
                issues.append(it)
        page_info = data["data"]["issues"]["pageInfo"]
        if page_info.get("hasNextPage"):
            cursor = page_info.get("endCursor")
            time.sleep(0.2)
        else:
            break
        # safety
        if page > 1000:
            errors.append({"page": page, "error": "Exceeded page safety limit (1000). Stopping."})
            break
    return issues, errors

# -------------------------
# EDA generator (fields list, payload list) -> CSV
# -------------------------
def generate_eda_csv(source_name: str, fields: list, payload: list, suggested_handling_map: dict, out_csv: Path):
    total_rows = len(payload)
    results = []
    timestamp_parse_errors = Counter()
    for field in fields:
        null_count = 0
        unique_vals = set()
        max_array_len = 0
        parse_errors = 0
        parsed_dates = []
        for obj in payload:
            if field not in obj:
                null_count += 1
                continue
            val = obj.get(field)
            # treat empty string/list/dict as null-like
            if val is None or (isinstance(val, str) and val.strip() == "") or (isinstance(val, (list, dict)) and len(val) == 0):
                null_count += 1
                continue
            # unique
            try:
                if isinstance(val, (dict, list)):
                    unique_vals.add(json.dumps(val, sort_keys=True, ensure_ascii=False))
                else:
                    unique_vals.add(str(val))
            except Exception:
                unique_vals.add(str(val))
            # array length
            if isinstance(val, list):
                max_array_len = max(max_array_len, len(val))
            # timestamp parsing for known timestamp fields
            if field in ("created_at","updated_at","due_by","fr_due_by","nr_due_by","createdAt","updatedAt","archivedAt"):
                try:
                    dt = dateparse.parse(val) if isinstance(val, str) else None
                    if dt:
                        parsed_dates.append(dt)
                except Exception:
                    parse_errors += 1
                    timestamp_parse_errors[field] += 1
        missing_rate = (null_count / total_rows) if total_rows > 0 else None
        inferred_type = "array" if field in ("cc_emails","fwd_emails","reply_cc_emails","ticket_cc_emails","ticket_bcc_emails","to_emails","tags","labels") else (
                        "timestamp" if field in ("created_at","updated_at","due_by","fr_due_by","nr_due_by","createdAt","updatedAt","archivedAt") else (
                        "json" if field in ("custom_fields","source_info","structured_description") else (
                        "boolean" if field in ("fr_escalated","spam","is_escalated","nr_escalated","trashed") else (
                        "numeric" if field in ("priority","associated_tickets_count","sentiment_score","initial_sentiment_score","estimate","number") else "string"))))
        suggested = suggested_handling_map.get(field, "")
        min_date = min(parsed_dates).isoformat() if parsed_dates else ""
        max_date = max(parsed_dates).isoformat() if parsed_dates else ""
        results.append({
            "Source": source_name,
            "Field": field,
            "Inferred Type": inferred_type,
            "Missing Rate": round(missing_rate, 6) if missing_rate is not None else "",
            "Unique Count": len(unique_vals),
            "Max Array Len": max_array_len if max_array_len > 0 else "",
            "Parse Errors": parse_errors,
            "Min Date": min_date,
            "Max Date": max_date,
            "Suggested Handling": suggested,
            "Notes": ""
        })
    # write CSV
    out_cols = ["Source","Field","Inferred Type","Missing Rate","Unique Count","Max Array Len","Parse Errors","Min Date","Max Date","Suggested Handling","Notes"]
    tmp = out_csv.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=out_cols)
        writer.writeheader()
        for r in results:
            writer.writerow(r)
    tmp.replace(out_csv)
    return {"total_rows": total_rows, "timestamp_parse_errors": dict(timestamp_parse_errors)}

# -------------------------
# Main run: weekly raw files, append to master CSVs, final EDA
# -------------------------
def run_weekly_backfill():
    run_start = utc_now_ts()
    ts = ts_for_filename(run_start)
    log = {
        "start": run_start.isoformat(),
        "range_start": START_DATE.isoformat(),
        "range_end": END_DATE.isoformat(),
        "freshdesk": {"weekly": [], "fetched_total": 0, "api_errors": []},
        "linear": {"weekly": [], "fetched_total": 0, "api_errors": []},
        "errors": []
    }

    # Read API keys
    try:
        fresh_key = read_api_key(FRESHDESK_API_KEY_PATH)
    except Exception as e:
        raise RuntimeError(f"Freshdesk API key error: {e}")
    try:
        linear_key = read_api_key(LINEAR_API_KEY_PATH)
    except Exception as e:
        raise RuntimeError(f"Linear API key error: {e}")

    # Master CSV paths
    master_fd_csv = DATA_DIR / "freshdesk" / f"freshdesk_tickets_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.csv"
    master_ln_csv = DATA_DIR / "linear" / f"linear_issues_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.csv"

    # Iterate weekly windows
    all_fd_payload = []
    all_fd_errors = []
    all_ln_payload = []
    all_ln_errors = []

    for w_start, w_end in week_windows(START_DATE, END_DATE):
        week_label = f"{w_start.strftime('%Y%m%d')}_{w_end.strftime('%Y%m%d')}"
        print(f"Processing week {week_label} ...")
        # --- Freshdesk per-week fetch ---
        try:
            fd_block, fd_errs = fetch_freshdesk_tickets(fresh_key, w_start, w_end)
            all_fd_payload.extend(fd_block)
            if fd_errs:
                all_fd_errors.extend(fd_errs)
            # Save raw JSON per week
            raw_fd_path = RAW_DIR / "freshdesk" / f"freshdesk_raw_{ts}_week_{week_label}.json"
            atomic_write_text(raw_fd_path, json.dumps(fd_block, indent=2, ensure_ascii=False))
            log["freshdesk"]["weekly"].append({"week": week_label, "raw_path": str(raw_fd_path), "count": len(fd_block), "errors": fd_errs})
            print(f"  Freshdesk: fetched {len(fd_block)} tickets for week {week_label}")
            # Normalize and append to master CSV
            snapshot_ts_iso = run_start.isoformat()
            fd_rows = [normalize_freshdesk_ticket(t, snapshot_ts_iso, raw_fd_path) for t in fd_block]
            if fd_rows:
                df_fd = pd.DataFrame(fd_rows)
                append_csv_atomic(master_fd_csv, df_fd)
            log["freshdesk"]["fetched_total"] += len(fd_block)
        except Exception as e:
            tb = traceback.format_exc()
            log["errors"].append({"system":"freshdesk_week", "week": week_label, "error": str(e), "traceback": tb})
            print(f"  Freshdesk week error: {e}")

        # --- Linear per-week fetch ---
        try:
            ln_block, ln_errs = fetch_linear_issues(linear_key, w_start, w_end)
            all_ln_payload.extend(ln_block)
            if ln_errs:
                all_ln_errors.extend(ln_errs)
            # Save raw JSON per week
            raw_ln_path = RAW_DIR / "linear" / f"linear_raw_{ts}_week_{week_label}.json"
            atomic_write_text(raw_ln_path, json.dumps(ln_block, indent=2, ensure_ascii=False))
            log["linear"]["weekly"].append({"week": week_label, "raw_path": str(raw_ln_path), "count": len(ln_block), "errors": ln_errs})
            print(f"  Linear: fetched {len(ln_block)} issues for week {week_label}")
            # Normalize and append to master CSV
            snapshot_ts_iso = run_start.isoformat()
            ln_rows = [normalize_linear_issue(it, snapshot_ts_iso, raw_ln_path) for it in ln_block]
            if ln_rows:
                df_ln = pd.DataFrame(ln_rows)
                append_csv_atomic(master_ln_csv, df_ln)
            log["linear"]["fetched_total"] += len(ln_block)
        except Exception as e:
            tb = traceback.format_exc()
            log["errors"].append({"system":"linear_week", "week": week_label, "error": str(e), "traceback": tb})
            print(f"  Linear week error: {e}")

        # small pause between weeks to be polite to APIs
        time.sleep(0.5)

    # Deduplicate aggregated payloads for EDA (by id) to avoid skewed EDA counts
    # For Freshdesk: dedupe by ticket id
    unique_fd = []
    seen_fd = set()
    for t in all_fd_payload:
        tid = t.get("id")
        if tid is None:
            unique_fd.append(t)
        elif tid not in seen_fd:
            seen_fd.add(tid)
            unique_fd.append(t)
    # For Linear: dedupe by id
    unique_ln = []
    seen_ln = set()
    for it in all_ln_payload:
        iid = it.get("id")
        if iid is None:
            unique_ln.append(it)
        elif iid not in seen_ln:
            seen_ln.add(iid)
            unique_ln.append(it)

    # Save final EDA CSVs (aggregated across full range)
    try:
        eda_fd_csv = DATA_DIR / f"eda_freshdesk_fields_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.csv"
        eda_fd_summary = generate_eda_csv("Freshdesk", FRESHDESK_FIELDS, unique_fd, SUGGESTED_HANDLING_FD, eda_fd_csv)
        log["freshdesk"]["eda_csv"] = str(eda_fd_csv)
        log["freshdesk"]["eda_summary"] = eda_fd_summary
        log["freshdesk"]["api_errors"] = all_fd_errors
        log["freshdesk"]["duplicate_id_count"] = max(0, len(all_fd_payload) - len(unique_fd))
    except Exception as e:
        tb = traceback.format_exc()
        log["errors"].append({"system":"freshdesk_eda", "error": str(e), "traceback": tb})

    try:
        eda_ln_csv = DATA_DIR / f"eda_linear_fields_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.csv"
        eda_ln_summary = generate_eda_csv("Linear", LINEAR_FIELDS, unique_ln, SUGGESTED_HANDLING_LN, eda_ln_csv)
        log["linear"]["eda_csv"] = str(eda_ln_csv)
        log["linear"]["eda_summary"] = eda_ln_summary
        log["linear"]["api_errors"] = all_ln_errors
        log["linear"]["duplicate_id_count"] = max(0, len(all_ln_payload) - len(unique_ln))
    except Exception as e:
        tb = traceback.format_exc()
        log["errors"].append({"system":"linear_eda", "error": str(e), "traceback": tb})

    # Save run log
    log_path = LOGS_DIR / f"snapshot_eda_{ts}.json"
    try:
        atomic_write_text(log_path, json.dumps(log, indent=2, ensure_ascii=False))
    except Exception as e:
        print("Failed to write run log:", e)

    # Print summary
    print("Backfill complete.")
    print(json.dumps({
        "start": log["start"],
        "range_start": log["range_start"],
        "range_end": log["range_end"],
        "freshdesk_fetched_total": log["freshdesk"]["fetched_total"],
        "linear_fetched_total": log["linear"]["fetched_total"],
        "freshdesk_eda_csv": log.get("freshdesk", {}).get("eda_csv"),
        "linear_eda_csv": log.get("linear", {}).get("eda_csv"),
        "log_path": str(log_path)
    }, indent=2, ensure_ascii=False))

    return log

# -------------------------
# Execute
# -------------------------
if __name__ == "__main__":
    try:
        result = run_weekly_backfill()
    except Exception as exc:
        print("Fatal error:", str(exc))
        print(traceback.format_exc())
        # attempt to write minimal run log
        try:
            ts = ts_for_filename()
            err_log = {"start": utc_now_ts().isoformat(), "fatal_error": str(exc)}
            atomic_write_text(LOGS_DIR / f"snapshot_eda_error_{ts}.json", json.dumps(err_log, indent=2, ensure_ascii=False))
        except Exception:
            pass
        raise


Processing week 20250601_20250607 ...
  Freshdesk: fetched 0 tickets for week 20250601_20250607
  Linear: fetched 0 issues for week 20250601_20250607
Processing week 20250608_20250614 ...
  Freshdesk: fetched 0 tickets for week 20250608_20250614
  Linear: fetched 0 issues for week 20250608_20250614
Processing week 20250615_20250621 ...


KeyboardInterrupt: 

Koden ovan gav inga data - fel på filtrering och för mycket på en gång. Testar att hämta en veckas data. 

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Backfill – TESTKÖRNING
Hämtar EN vecka och stoppar.
"""

import requests
import json
import time
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dateutil import parser as dateparse
import pandas as pd
import traceback

# -------------------------
# CONFIG
# -------------------------
PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "raw"
CREDENTIALS_DIR = PROJECT_ROOT / "credentials"

LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"
LINEAR_API_URL = "https://api.linear.app/graphql"
LINEAR_PAGE_SIZE = 100

FRESHDESK_API_KEY_PATH = CREDENTIALS_DIR / "Freshdesk_API-key.txt"
FRESHDESK_DOMAIN = "intersolia.freshdesk.com"
FRESHDESK_TICKETS_ENDPOINT = f"https://{FRESHDESK_DOMAIN}/api/v2/tickets"
FRESHDESK_PAGE_SIZE = 100

# TEST: stop after first week
STOP_AFTER_FIRST_WEEK = True

# Date range (inclusive)
START_DATE = datetime(2025, 6, 1, 0, 0, 0, tzinfo=timezone.utc)
END_DATE = datetime(2026, 5, 31, 23, 59, 59, tzinfo=timezone.utc)

# Ensure dirs
for p in [RAW_DIR, RAW_DIR / "freshdesk", RAW_DIR / "linear"]:
    p.mkdir(parents=True, exist_ok=True)

# Helpers
def utc_now_ts():
    return datetime.now(timezone.utc)

def ts_for_filename(dt=None):
    if dt is None:
        dt = utc_now_ts()
    return dt.strftime("%Y%m%dT%H%M%SZ")

def atomic_write_text(path: Path, text: str, encoding="utf-8"):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding=encoding)
    tmp.replace(path)

def read_api_key(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing API key file: {path}")
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API key file {path} is empty")
    return key

def http_post_with_retry(url, headers=None, json_payload=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.post(url, headers=headers, json=json_payload, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

def http_get_with_retry(url, auth=None, params=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.get(url, auth=auth, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

# -------------------------
# Week generator
# -------------------------
def week_windows(start_dt: datetime, end_dt: datetime):
    cur = start_dt
    while cur <= end_dt:
        w_start = cur
        w_end = min(cur + timedelta(days=6, hours=23, minutes=59, seconds=59), end_dt)
        yield (w_start, w_end)
        cur = w_end + timedelta(seconds=1)

# -------------------------
# Fetch Freshdesk
# -------------------------
def fetch_freshdesk_tickets(api_key: str, start_dt: datetime, end_dt: datetime):
    auth = (api_key, "X")
    tickets = []
    page = 1
    params = {
        "per_page": FRESHDESK_PAGE_SIZE,
        "page": page,
        "updated_since": start_dt.isoformat().replace("+00:00", "Z")
    }

    while True:
        params["page"] = page
        try:
            resp = http_get_with_retry(FRESHDESK_TICKETS_ENDPOINT, auth=auth, params=params)
            block = resp.json()
        except Exception:
            break

        if not isinstance(block, list):
            break

        tickets.extend(block)

        if len(block) < FRESHDESK_PAGE_SIZE:
            break

        page += 1
        if page > 200:
            break

    # local filter
    filtered = []
    for t in tickets:
        created = None
        updated = None
        try:
            created = dateparse.parse(t.get("created_at")) if t.get("created_at") else None
        except:
            pass
        try:
            updated = dateparse.parse(t.get("updated_at")) if t.get("updated_at") else None
        except:
            pass

        if (created and start_dt <= created <= end_dt) or (updated and start_dt <= updated <= end_dt):
            filtered.append(t)

    return filtered

# -------------------------
# Fetch Linear
# -------------------------
def fetch_linear_issues(api_key: str, start_dt: datetime, end_dt: datetime):
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    sel = """
        id
        number
        identifier
        title
        description
        createdAt
        updatedAt
        archivedAt
        priority
        estimate
        trashed
        state { id name type }
        assignee { id name email }
        project { id name }
        team { id name }
        labels(first:50) { nodes { id name } }
    """

    query = f"""
    query($first:Int!, $after:String) {{
      issues(first:$first, after:$after) {{
        nodes {{
          {sel}
        }}
        pageInfo {{ hasNextPage endCursor }}
      }}
    }}
    """

    issues = []
    cursor = None
    page = 0

    while True:
        page += 1
        payload = {"query": query, "variables": {"first": LINEAR_PAGE_SIZE, "after": cursor}}

        try:
            resp = http_post_with_retry(LINEAR_API_URL, headers=headers, json_payload=payload)
            data = resp.json()
        except Exception:
            break

        if "errors" in data:
            break

        block = data["data"]["issues"]["nodes"]

        for it in block:
            created = None
            updated = None
            try:
                created = dateparse.parse(it.get("createdAt")) if it.get("createdAt") else None
            except:
                pass
            try:
                updated = dateparse.parse(it.get("updatedAt")) if it.get("updatedAt") else None
            except:
                pass

            if (created and start_dt <= created <= end_dt) or (updated and start_dt <= updated <= end_dt):
                issues.append(it)

        page_info = data["data"]["issues"]["pageInfo"]
        if page_info.get("hasNextPage"):
            cursor = page_info.get("endCursor")
        else:
            break

        if page > 300:
            break

    return issues

# -------------------------
# MAIN – only first week
# -------------------------
def run_test_week():
    fresh_key = read_api_key(FRESHDESK_API_KEY_PATH)
    linear_key = read_api_key(LINEAR_API_KEY_PATH)

    ts = ts_for_filename()

    for w_start, w_end in week_windows(START_DATE, END_DATE):
        week_label = f"{w_start.strftime('%Y%m%d')}_{w_end.strftime('%Y%m%d')}"
        print(f"\n=== HÄMTAR VECKA {week_label} ===")

        # Freshdesk
        fd = fetch_freshdesk_tickets(fresh_key, w_start, w_end)
        raw_fd_path = RAW_DIR / "freshdesk" / f"freshdesk_raw_{ts}_week_{week_label}.json"
        atomic_write_text(raw_fd_path, json.dumps(fd, indent=2, ensure_ascii=False))

        # Linear
        ln = fetch_linear_issues(linear_key, w_start, w_end)
        raw_ln_path = RAW_DIR / "linear" / f"linear_raw_{ts}_week_{week_label}.json"
        atomic_write_text(raw_ln_path, json.dumps(ln, indent=2, ensure_ascii=False))

        print(f"\n=== RESULTAT VECKA {week_label} ===")
        print(f"Freshdesk: {len(fd)} tickets")
        print(f"Linear:    {len(ln)} issues")
        print(f"Råfiler sparade i raw/<system>/")

        if STOP_AFTER_FIRST_WEEK:
            print("\nSTOP_AFTER_FIRST_WEEK = True → Avslutar efter första veckan.")
            return

        time.sleep(0.5)

if __name__ == "__main__":
    run_test_week()



=== HÄMTAR VECKA 20250601_20250607 ===

=== RESULTAT VECKA 20250601_20250607 ===
Freshdesk: 0 tickets
Linear:    0 issues
Råfiler sparade i raw/<system>/

STOP_AFTER_FIRST_WEEK = True → Avslutar efter första veckan.


Freshdesk vill inte släppa ifrån sig data äldre än 30 dagar på det här sättet. Vi får köra en export i stället. 

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Freshdesk Export Backfill
Hämtar ALLA tickets via Export API och filtrerar lokalt.
"""

import requests
import time
import zipfile
import io
import json
from pathlib import Path
from datetime import datetime, timezone
from dateutil import parser as dateparse

# -------------------------
# CONFIG
# -------------------------
PROJECT_ROOT = Path(".").resolve()
EXPORT_DIR = PROJECT_ROOT / "raw" / "freshdesk_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_DIR = PROJECT_ROOT / "credentials"
FRESHDESK_API_KEY_PATH = CREDENTIALS_DIR / "Freshdesk_API-key.txt"

FRESHDESK_DOMAIN = "intersolia.freshdesk.com"
EXPORT_ENDPOINT = f"https://{FRESHDESK_DOMAIN}/api/v2/export/tickets"

START_DATE = datetime(2025, 6, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2026, 5, 31, 23, 59, 59, tzinfo=timezone.utc)

# -------------------------
# Helpers
# -------------------------
def read_api_key(path: Path):
    key = path.read_text().strip()
    if not key:
        raise ValueError("Freshdesk API key file is empty")
    return key

def create_export_job(api_key: str):
    print("Skapar export-jobb...")
    resp = requests.post(
        EXPORT_ENDPOINT,
        auth=(api_key, "X"),
        json={"include": "all"},
        timeout=30
    )
    resp.raise_for_status()
    job = resp.json()
    return job["id"]

def poll_export_job(api_key: str, job_id: str):
    print(f"Väntar på export-jobb {job_id} ...")
    status_url = f"{EXPORT_ENDPOINT}/{job_id}"

    while True:
        resp = requests.get(status_url, auth=(api_key, "X"), timeout=30)
        resp.raise_for_status()
        data = resp.json()

        state = data.get("status")
        print(f"  Status: {state}")

        if state == "completed":
            return data["download_url"]

        if state == "failed":
            raise RuntimeError("Freshdesk export-jobb misslyckades")

        time.sleep(5)

def download_export_zip(api_key: str, url: str):
    print("Laddar ner ZIP...")
    resp = requests.get(url, auth=(api_key, "X"), timeout=60)
    resp.raise_for_status()
    return resp.content

def extract_zip_to_json(zip_bytes: bytes):
    print("Extraherar ZIP...")
    z = zipfile.ZipFile(io.BytesIO(zip_bytes))
    extracted = []

    for name in z.namelist():
        if name.endswith(".json"):
            with z.open(name) as f:
                data = json.loads(f.read().decode("utf-8"))
                extracted.extend(data)

    return extracted

def filter_by_date(tickets):
    print("Filtrerar på datum...")
    out = []
    for t in tickets:
        created = None
        updated = None
        try:
            created = dateparse.parse(t.get("created_at")) if t.get("created_at") else None
        except:
            pass
        try:
            updated = dateparse.parse(t.get("updated_at")) if t.get("updated_at") else None
        except:
            pass

        if (created and START_DATE <= created <= END_DATE) or \
           (updated and START_DATE <= updated <= END_DATE):
            out.append(t)

    return out

# -------------------------
# MAIN
# -------------------------
def run_export_backfill():
    api_key = read_api_key(FRESHDESK_API_KEY_PATH)

    job_id = create_export_job(api_key)
    download_url = poll_export_job(api_key, job_id)
    zip_bytes = download_export_zip(api_key, download_url)

    all_tickets = extract_zip_to_json(zip_bytes)
    print(f"Totalt antal tickets i exporten: {len(all_tickets)}")

    filtered = filter_by_date(all_tickets)
    print(f"Tickets inom intervallet: {len(filtered)}")

    out_path = EXPORT_DIR / "freshdesk_export_filtered_20250601_20260531.json"
    out_path.write_text(json.dumps(filtered, indent=2, ensure_ascii=False))

    print(f"Klart! Sparat till: {out_path}")

if __name__ == "__main__":
    run_export_backfill()


Skapar export-jobb...


HTTPError: 404 Client Error: Not Found for url: https://intersolia.freshdesk.com/api/v2/export/tickets

Nehej, det gick inte heller. Den exporten får göras manuellt då.

Exportera Freshdesk Tickets via UI

1. Gå till Freshdesk

2. Klicka Admin

3. Under Account → välj Data Export

4. Välj Tickets

5. Välj All time (går inte att ta ut bara ett år)

6. Klicka Export

Ska ge en ZIP.

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
30-dagars snapshot för Freshdesk och Linear
Perfekt för EDA, brons och silver.
"""

import requests
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dateutil import parser as dateparse

# -------------------------
# CONFIG
# -------------------------
PROJECT_ROOT = Path(".").resolve()
OUT_DIR = PROJECT_ROOT / "raw" / "snapshot_30d"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_DIR = PROJECT_ROOT / "credentials"
FRESHDESK_API_KEY_PATH = CREDENTIALS_DIR / "Freshdesk_API-key.txt"
LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"

FRESHDESK_DOMAIN = "intersolia.freshdesk.com"
FRESHDESK_TICKETS_ENDPOINT = f"https://{FRESHDESK_DOMAIN}/api/v2/tickets"

LINEAR_API_URL = "https://api.linear.app/graphql"

# 30 dagar bakåt
END = datetime.now(timezone.utc)
START = END - timedelta(days=30)

# -------------------------
# Helpers
# -------------------------
def read_api_key(path: Path):
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API key file is empty: {path}")
    return key

# -------------------------
# Freshdesk 30 dagar
# -------------------------
def fetch_freshdesk_30d(api_key):
    auth = (api_key, "X")
    page = 1
    out = []

    params = {
        "per_page": 100,
        "page": page,
        "updated_since": START.isoformat().replace("+00:00", "Z")
    }

    print("  Freshdesk: startar paging...")

    while True:
        params["page"] = page
        r = requests.get(FRESHDESK_TICKETS_ENDPOINT, auth=auth, params=params)

        if r.status_code == 400:
            print("  Freshdesk: 400 Bad Request – troligen slut på data.")
            break

        r.raise_for_status()

        block = r.json()
        if not isinstance(block, list):
            print("  Freshdesk: oväntat svar, avbryter.")
            break

        print(f"  Freshdesk: sida {page}, {len(block)} tickets")

        out.extend(block)

        if len(block) < 100:
            break

        page += 1

    # lokal filtrering
    filtered = []
    for t in out:
        created = dateparse.parse(t["created_at"]) if t.get("created_at") else None
        updated = dateparse.parse(t["updated_at"]) if t.get("updated_at") else None

        if (created and START <= created <= END) or (updated and START <= updated <= END):
            filtered.append(t)

    return filtered

# -------------------------
# Linear 30 dagar
# -------------------------
def fetch_linear_30d(api_key):
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
    }

    query = """
    query($first:Int!, $after:String) {
      issues(first:$first, after:$after) {
        nodes {
          id
          number
          title
          createdAt
          updatedAt
          state { id name }
          assignee { id name email }
        }
        pageInfo { hasNextPage endCursor }
      }
    }
    """

    issues = []
    cursor = None
    page = 0

    print("  Linear: startar GraphQL paging...")

    while True:
        page += 1
        payload = {"query": query, "variables": {"first": 100, "after": cursor}}
        r = requests.post(LINEAR_API_URL, headers=headers, json=payload)

        try:
            data = r.json()
        except Exception:
            print("  Linear: kunde inte tolka JSON-svar.")
            break

        if "errors" in data:
            print("  Linear GraphQL error:", data["errors"])
            break

        block = data["data"]["issues"]["nodes"]
        print(f"  Linear: sida {page}, {len(block)} issues")

        for it in block:
            created = dateparse.parse(it["createdAt"]) if it.get("createdAt") else None
            updated = dateparse.parse(it["updatedAt"]) if it.get("updatedAt") else None

            if (created and START <= created <= END) or (updated and START <= updated <= END):
                issues.append(it)

        pageInfo = data["data"]["issues"]["pageInfo"]
        if pageInfo["hasNextPage"]:
            cursor = pageInfo["endCursor"]
        else:
            break

    return issues

# -------------------------
# MAIN
# -------------------------
def run_snapshot():
    fd_key = read_api_key(FRESHDESK_API_KEY_PATH)
    ln_key = read_api_key(LINEAR_API_KEY_PATH)

    print("Hämtar Freshdesk 30 dagar...")
    fd = fetch_freshdesk_30d(fd_key)
    print(f"Freshdesk: {len(fd)} tickets")

    print("Hämtar Linear 30 dagar...")
    ln = fetch_linear_30d(ln_key)
    print(f"Linear: {len(ln)} issues")

    # UTF-8-säker skrivning
    (OUT_DIR / "freshdesk_30d.json").write_bytes(
        json.dumps(fd, indent=2, ensure_ascii=False).encode("utf-8")
    )
    (OUT_DIR / "linear_30d.json").write_bytes(
        json.dumps(ln, indent=2, ensure_ascii=False).encode("utf-8")
    )

    print("Klart! Data sparat i raw/snapshot_30d/")

if __name__ == "__main__":
    run_snapshot()


Hämtar Freshdesk 30 dagar...
  Freshdesk: startar paging...
  Freshdesk: sida 1, 100 tickets
  Freshdesk: sida 2, 100 tickets
  Freshdesk: sida 3, 100 tickets
  Freshdesk: sida 4, 100 tickets
  Freshdesk: sida 5, 100 tickets
  Freshdesk: sida 6, 100 tickets
  Freshdesk: sida 7, 100 tickets
  Freshdesk: sida 8, 100 tickets
  Freshdesk: sida 9, 100 tickets
  Freshdesk: sida 10, 100 tickets
  Freshdesk: sida 11, 100 tickets
  Freshdesk: sida 12, 100 tickets
  Freshdesk: sida 13, 100 tickets
  Freshdesk: sida 14, 100 tickets
  Freshdesk: sida 15, 100 tickets
  Freshdesk: sida 16, 100 tickets
  Freshdesk: sida 17, 100 tickets
  Freshdesk: sida 18, 100 tickets
  Freshdesk: sida 19, 100 tickets
  Freshdesk: sida 20, 100 tickets
  Freshdesk: sida 21, 100 tickets
  Freshdesk: sida 22, 100 tickets
  Freshdesk: sida 23, 100 tickets
  Freshdesk: sida 24, 100 tickets
  Freshdesk: sida 25, 100 tickets
  Freshdesk: sida 26, 100 tickets
  Freshdesk: sida 27, 100 tickets
  Freshdesk: sida 28, 100 ticke

Korrigerad snapshot för Linear (komplett)

In [16]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Hämtar 30 dagars Linear-data med FULLT schema enligt Linear-dokumentationen.
Flattenar nested GraphQL-fält till samma struktur som i fredags.
Sparar resultatet i raw/snapshot_30d/linear_30d_full.json
"""

import requests
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dateutil import parser as dateparse

# ---------------------------------------------------------
# KONFIGURATION
# ---------------------------------------------------------
PROJECT_ROOT = Path(".").resolve()
OUT_DIR = PROJECT_ROOT / "raw" / "snapshot_30d"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_DIR = PROJECT_ROOT / "credentials"
LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"

LINEAR_API_URL = "https://api.linear.app/graphql"

# 30 dagar bakåt
END = datetime.now(timezone.utc)
START = END - timedelta(days=30)


# ---------------------------------------------------------
# HJÄLPFUNKTIONER
# ---------------------------------------------------------
def read_api_key(path: Path):
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API-nyckeln i {path} är tom.")
    return key


def safe(obj, key, default=None):
    """Säker hämtning från nested dicts."""
    if obj is None:
        return default
    return obj.get(key, default)


def flatten_issue(issue: dict) -> dict:
    """Flattenar nested GraphQL-fält till samma struktur som i fredags."""

    state = issue.get("state")
    assignee = issue.get("assignee")
    project = issue.get("project")
    team = issue.get("team")
    labels = issue.get("labels", {}).get("nodes", [])

    return {
        "id": issue.get("id"),
        "number": issue.get("number"),
        "identifier": issue.get("identifier"),
        "title": issue.get("title"),
        "description": issue.get("description"),
        "createdAt": issue.get("createdAt"),
        "updatedAt": issue.get("updatedAt"),
        "archivedAt": issue.get("archivedAt"),
        "priority": issue.get("priority"),
        "estimate": issue.get("estimate"),
        "trashed": issue.get("trashed"),

        # state
        "state_id": safe(state, "id"),
        "state_name": safe(state, "name"),
        "state_type": safe(state, "type"),

        # assignee
        "assignee_id": safe(assignee, "id"),
        "assignee_name": safe(assignee, "name"),
        "assignee_email": safe(assignee, "email"),

        # project
        "project_id": safe(project, "id"),
        "project_name": safe(project, "name"),

        # team
        "team_id": safe(team, "id"),
        "team_name": safe(team, "name"),

        # labels
        "labels": [lbl.get("name") for lbl in labels if lbl.get("name")]
    }


# ---------------------------------------------------------
# HÄMTA LINEAR-DATA (FULLT SCHEMA)
# ---------------------------------------------------------
def fetch_linear_30d(api_key: str):
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
    }

    query = """
    query($first:Int!, $after:String) {
      issues(first:$first, after:$after) {
        nodes {
          id
          number
          identifier
          title
          description
          createdAt
          updatedAt
          archivedAt
          priority
          estimate
          trashed
          state { id name type }
          assignee { id name email }
          project { id name }
          team { id name }
          labels {
            nodes {
              name
            }
          }
        }
        pageInfo { hasNextPage endCursor }
      }
    }
    """

    issues = []
    cursor = None
    page = 0

    print("=== Startar Linear GraphQL-hämtning (fullt schema) ===")

    while True:
        page += 1
        payload = {"query": query, "variables": {"first": 100, "after": cursor}}

        r = requests.post(LINEAR_API_URL, headers=headers, json=payload)
        data = r.json()

        if "errors" in data:
            print("GraphQL error:", data["errors"])
            break

        block = data["data"]["issues"]["nodes"]
        print(f"Sida {page}: {len(block)} issues")

        # Filtrera på 30 dagar
        for it in block:
            created = dateparse.parse(it["createdAt"]) if it.get("createdAt") else None
            updated = dateparse.parse(it["updatedAt"]) if it.get("updatedAt") else None

            if (created and START <= created <= END) or (updated and START <= updated <= END):
                issues.append(flatten_issue(it))

        pageInfo = data["data"]["issues"]["pageInfo"]
        if pageInfo["hasNextPage"]:
            cursor = pageInfo["endCursor"]
        else:
            break

    print(f"Totalt matchande issues (30 dagar): {len(issues)}")
    return issues


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
def run_linear_snapshot():
    api_key = read_api_key(LINEAR_API_KEY_PATH)

    print("Hämtar Linear 30 dagar (full schema)...")
    ln = fetch_linear_30d(api_key)

    out_path = OUT_DIR / "linear_30d_full.json"
    out_path.write_bytes(
        json.dumps(ln, indent=2, ensure_ascii=False).encode("utf-8")
    )

    print(f"Klart! Sparat till: {out_path}")


# ---------------------------------------------------------
# ENTRYPOINT
# ---------------------------------------------------------
if __name__ == "__main__":
    run_linear_snapshot()


Hämtar Linear 30 dagar (full schema)...
=== Startar Linear GraphQL-hämtning (fullt schema) ===
Sida 1: 100 issues
Sida 2: 100 issues
Sida 3: 100 issues
Sida 4: 100 issues
Sida 5: 100 issues
Sida 6: 100 issues
Sida 7: 100 issues
Sida 8: 100 issues
Sida 9: 100 issues
Sida 10: 100 issues
Sida 11: 38 issues
Totalt matchande issues (30 dagar): 359
Klart! Sparat till: C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\raw\snapshot_30d\linear_30d_full.json


Sådär. 30 dagar snapshot att experimentera med. 

In [18]:
import pandas as pd
import json

fd = pd.read_json("raw/snapshot_30d/freshdesk_30d.json")
ln = pd.read_json("raw/snapshot_30d/linear_30d_full.json")

print(fd.shape)
print(ln.shape)


(9228, 39)
(359, 22)


Skapa CSV från json-filerna.

In [10]:
import pandas as pd

fd = pd.read_json("raw/snapshot_30d/freshdesk_30d.json")

eda_fd = fd.describe(include="all").transpose()

eda_fd.to_csv("raw/snapshot_30d/freshdesk_eda_describe.csv", encoding="utf-8")


In [19]:
ln = pd.read_json("raw/snapshot_30d/linear_30d_full.json")

eda_ln = ln.describe(include="all").transpose()

eda_ln.to_csv("raw/snapshot_30d/linear_eda_describe.csv", encoding="utf-8")


Hämta alla Linear från 2025-01-01


In [21]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Hämtar Linear-data med FULLT schema enligt Linear-dokumentationen.
Flattenar nested GraphQL-fält.
Sparar resultatet i raw/linear_2025-01-01-2026-06-01.json
"""

import requests
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dateutil import parser as dateparse

# ---------------------------------------------------------
# KONFIGURATION
# ---------------------------------------------------------
PROJECT_ROOT = Path(".").resolve()
OUT_DIR = PROJECT_ROOT / "raw"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_DIR = PROJECT_ROOT / "credentials"
LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"

LINEAR_API_URL = "https://api.linear.app/graphql"

# 30 dagar bakåt
END = datetime.now(timezone.utc)
START = datetime(2025, 1, 1, tzinfo=timezone.utc)  # START satt till 2025-01-01 för att få mer data


# ---------------------------------------------------------
# HJÄLPFUNKTIONER
# ---------------------------------------------------------
def read_api_key(path: Path):
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API-nyckeln i {path} är tom.")
    return key


def safe(obj, key, default=None):
    """Säker hämtning från nested dicts."""
    if obj is None:
        return default
    return obj.get(key, default)


def flatten_issue(issue: dict) -> dict:
    """Flattenar nested GraphQL-fält."""

    state = issue.get("state")
    assignee = issue.get("assignee")
    project = issue.get("project")
    team = issue.get("team")
    labels = issue.get("labels", {}).get("nodes", [])

    return {
        "id": issue.get("id"),
        "number": issue.get("number"),
        "identifier": issue.get("identifier"),
        "title": issue.get("title"),
        "description": issue.get("description"),
        "createdAt": issue.get("createdAt"),
        "updatedAt": issue.get("updatedAt"),
        "archivedAt": issue.get("archivedAt"),
        "priority": issue.get("priority"),
        "estimate": issue.get("estimate"),
        "trashed": issue.get("trashed"),

        # state
        "state_id": safe(state, "id"),
        "state_name": safe(state, "name"),
        "state_type": safe(state, "type"),

        # assignee
        "assignee_id": safe(assignee, "id"),
        "assignee_name": safe(assignee, "name"),
        "assignee_email": safe(assignee, "email"),

        # project
        "project_id": safe(project, "id"),
        "project_name": safe(project, "name"),

        # team
        "team_id": safe(team, "id"),
        "team_name": safe(team, "name"),

        # labels
        "labels": [lbl.get("name") for lbl in labels if lbl.get("name")]
    }


# ---------------------------------------------------------
# HÄMTA LINEAR-DATA (FULLT SCHEMA)
# ---------------------------------------------------------
def fetch_linear_full(api_key: str):
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
    }

    query = """
    query($first:Int!, $after:String) {
      issues(first:$first, after:$after, includeArchived:true) {
        nodes {
          id
          number
          identifier
          title
          description
          createdAt
          updatedAt
          archivedAt
          priority
          estimate
          trashed
          state { id name type }
          assignee { id name email }
          project { id name }
          team { id name }
          labels {
            nodes {
              name
            }
          }
        }
        pageInfo { hasNextPage endCursor }
      }
    }
    """

    issues = []
    cursor = None
    page = 0

    print("=== Startar Linear GraphQL-hämtning (fullt schema) ===")

    while True:
        page += 1
        payload = {"query": query, "variables": {"first": 100, "after": cursor}}

        r = requests.post(LINEAR_API_URL, headers=headers, json=payload)
        data = r.json()

        if "errors" in data:
            print("GraphQL error:", data["errors"])
            break

        block = data["data"]["issues"]["nodes"]
        print(f"Sida {page}: {len(block)} issues")

        # Filtrera datum
        for it in block:
            created = dateparse.parse(it["createdAt"]) if it.get("createdAt") else None
            updated = dateparse.parse(it["updatedAt"]) if it.get("updatedAt") else None

            if (created and START <= created <= END) or (updated and START <= updated <= END):
                issues.append(flatten_issue(it))

        pageInfo = data["data"]["issues"]["pageInfo"]
        if pageInfo["hasNextPage"]:
            cursor = pageInfo["endCursor"]
        else:
            break

    print(f"Totalt matchande issues: {len(issues)}")
    return issues


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
def run_linear_snapshot():
    api_key = read_api_key(LINEAR_API_KEY_PATH)

    print("Hämtar Linear  (full schema)...")
    ln = fetch_linear_full(api_key)

    out_path = OUT_DIR / "linear_2025-01-01-2026-06-01.json"
    out_path.write_bytes(
        json.dumps(ln, indent=2, ensure_ascii=False).encode("utf-8")
    )

    print(f"Klart! Sparat till: {out_path}")


# ---------------------------------------------------------
# ENTRYPOINT
# ---------------------------------------------------------
if __name__ == "__main__":
    run_linear_snapshot()


Hämtar Linear  (full schema)...
=== Startar Linear GraphQL-hämtning (fullt schema) ===
Sida 1: 100 issues
Sida 2: 100 issues
Sida 3: 100 issues
Sida 4: 100 issues
Sida 5: 100 issues
Sida 6: 100 issues
Sida 7: 100 issues
Sida 8: 100 issues
Sida 9: 100 issues
Sida 10: 100 issues
Sida 11: 68 issues
Totalt matchande issues: 1068
Klart! Sparat till: C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\raw\linear_2025-01-01-2026-06-01.json


Kolla Freshdesk-status

In [22]:
import requests
import base64
import json
from pathlib import Path

# 1. Läs API-nyckeln från fil
api_key_path = Path("credentials/Freshdesk_API-key.txt")
api_key = api_key_path.read_text().strip()

# 2. Bygg Basic Auth-header
pair = f"{api_key}:X"
encoded = base64.b64encode(pair.encode("ascii")).decode("ascii")

headers = {
    "Authorization": f"Basic {encoded}"
}

# 3. Anropa Freshdesk API
url = "https://intersolia.freshdesk.com/api/v2/ticket_fields"
response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

# 4. Spara hela svaret till fil
output_path = Path("freshdesk_ticket_fields.json")
output_path.write_text(json.dumps(data, indent=2))

print("✔ Hämtat ticket_fields och sparat till freshdesk_ticket_fields.json")

# 5. Extrahera statuslistan (choices)
status_fields = [
    field for field in data 
    if field.get("name") == "status"
]

if status_fields:
    status_choices = status_fields[0].get("choices", {})
    print("\n✔ Status-ID → Status-namn:")
    for k, v in status_choices.items():
        print(f"{k}: {v}")
else:
    print("❗ Hittade ingen status-field i API-svaret.")


✔ Hämtat ticket_fields och sparat till freshdesk_ticket_fields.json

✔ Status-ID → Status-namn:
2: ['Open', 'Ticket is in progress']
3: ['Pending', 'Awaiting your answer']
4: ['Resolved', 'Ticket is resolved']
5: ['Closed', 'This ticket is closed']
10: ['Information needed (internal)', 'Ticket is waiting for internal info']
24: ['Customer responded', 'Customer Responded']
17: ['New case', 'Ticket is waiting for technical support']
6: ['Available', 'Ticket is waiting for technical support']
12: ['Investigate', 'Ticket is waiting for technical support']
8: ['In progress', 'Ticket is in progress']
9: ['In testing', 'Ticket is in testing']
7: ['Escalation', 'Ticket is escalated']
25: ['Task in backlog', 'Ticket is under review for future changes']
19: ['Need sprint planning', 'Ticket is under review for future changes']
14: ['Ready for hotfix', 'Ticket is ready for hotfix']
27: ['Product feedback', 'Product feedback']
16: ['Task done', 'Ticket is in progress']
26: ['Uninstall', 'Ticket is 

Provar att ta ned 18 månader Freshdesk-data. Flaskhals när API inte tål för stora mängder data. Splittar ned från hela tidsperioden till månad till vecka.

Tyvärr vägrar Freshdesk att släppa ifrån sig data i segment och det kraschar när datamängden blir för stor. 

In [26]:
import requests
import base64
import json
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

# ---------------------------------------------------------
# 1. Läs API-nyckeln
# ---------------------------------------------------------
api_key_path = Path("credentials/Freshdesk_API-key.txt")
api_key = api_key_path.read_text().strip()

pair = f"{api_key}:X"
encoded = base64.b64encode(pair.encode("ascii")).decode("ascii")

headers = {
    "Authorization": f"Basic {encoded}"
}

# ---------------------------------------------------------
# 2. Funktion: hämta tickets via updated_since + pagination
# ---------------------------------------------------------
def fetch_tickets_updated_since(start_date):
    url = "https://intersolia.freshdesk.com/api/v2/tickets"
    params = {
        "updated_since": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "per_page": 100
    }

    tickets = []
    session = requests.Session()
    session.headers.update(headers)

    while True:
        response = session.get(url, params=params)
        response.raise_for_status()

        data = response.json()
        tickets.extend(data)

        # Pagination via Link-header
        link = response.headers.get("link")
        if link and 'rel="next"' in link:
            next_url = link.split(";")[0].strip("<>")
            url = next_url
            params = {}  # nästa URL innehåller redan query params
        else:
            break

    return tickets

# ---------------------------------------------------------
# 3. Datumintervall
# ---------------------------------------------------------
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 5, 31)

raw_dir = Path("raw")
raw_dir.mkdir(exist_ok=True)

master = []

# ---------------------------------------------------------
# 4. Kör 7-dagars batcher
# ---------------------------------------------------------
current = START_DATE

while current <= END_DATE:
    batch_start = current
    batch_end = min(current + timedelta(days=7), END_DATE)

    print(f"⏳ Hämtar batch: {batch_start} → {batch_end}")

    tickets_raw = fetch_tickets_updated_since(batch_start)

    # Filtrera på created_at
    def parse_date(d):
        return datetime.strptime(d, "%Y-%m-%dT%H:%M:%SZ")

    filtered = []
    for t in tickets_raw:
        created = parse_date(t["created_at"])
        if batch_start <= created <= batch_end:
            filtered.append(t)

    print(f"✔ {len(filtered)} tickets i batchen")

    # Spara batch
    batch_file = raw_dir / f"freshdesk_{batch_start.strftime('%Y-%m-%d')}_{batch_end.strftime('%Y-%m-%d')}.json"
    batch_file.write_text(json.dumps(filtered, indent=2))

    master.extend(filtered)

    current = batch_end + timedelta(days=1)

# ---------------------------------------------------------
# 5. Spara masterfil
# ---------------------------------------------------------
master_file = raw_dir / "Freshdesk_2025-01-01_2026-05-31.json"
master_file.write_text(json.dumps(master, indent=2))

print(f"✔ Klar! Masterfil sparad till {master_file}")


⏳ Hämtar batch: 2025-01-01 00:00:00 → 2025-01-08 00:00:00


HTTPError: 400 Client Error: Bad Request for url: https://intersolia.freshdesk.com/api/v2/tickets?updated_since=2025-01-01T00:00:00Z&per_page=100&page=301

Kollar om min API-nyckel duger. Annars får jag be Carro om hjälp. 

In [27]:
import requests
import base64
from pathlib import Path

api_key = Path("credentials/Freshdesk_API-key.txt").read_text().strip()
pair = f"{api_key}:X"
encoded = base64.b64encode(pair.encode("ascii")).decode("ascii")

headers = {"Authorization": f"Basic {encoded}"}

url = "https://intersolia.freshdesk.com/api/v2/account"
response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text)


403
{"code":"access_denied","message":"You are not authorized to perform this action."}


Nähäpp.

Nytt försök att hämta Freshdesk-data. Kör en test med bara en "historisk" dag. 

In [32]:
#!/usr/bin/env python3
"""
Hämtar alla Freshdesk tickets från och med 2026-05-01 med list-tickets (updated_since).
Spara som fetch_from_20260501.py och kör: python fetch_from_20260501.py
Kräver: credentials/Freshdesk_API-key.txt med API-nyckeln, och internetåtkomst.
"""
import requests
import base64
import time
import json
from pathlib import Path
from datetime import datetime, timedelta

# ---------- Konfiguration ----------
API_KEY_FILE = Path("credentials/Freshdesk_API-key.txt")
DOMAIN = "intersolia"
UPDATED_SINCE = "2025-05-01"   # inclusive, YYYY-MM-DD
PER_PAGE = 100                  # list-tickets stödjer per_page
SLEEP_BETWEEN_REQUESTS = 0.25
MAX_RETRIES = 4
OUTPUT_FILE = Path("freshdesk_tickets_from_20260501.ndjson")
# -----------------------------------

def read_api_key(path: Path) -> str:
    return path.read_text(encoding="utf-8").strip()

def build_headers(api_key: str) -> dict:
    pair = f"{api_key}:X"
    encoded = base64.b64encode(pair.encode("ascii")).decode("ascii")
    return {"Authorization": f"Basic {encoded}", "Accept": "application/json"}

def fetch_page(session, headers, page, per_page, updated_since):
    url = f"https://{DOMAIN}.freshdesk.com/api/v2/tickets"
    params = {"page": page, "per_page": per_page, "updated_since": updated_since}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, headers=headers, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            elif r.status_code in (429, 503):
                wait = 2 ** attempt
                print(f"Rate/service ({r.status_code}). Backoff {wait}s (attempt {attempt}).")
                time.sleep(wait)
            elif r.status_code == 401:
                raise RuntimeError("401 Unauthorized — kontrollera API-nyckel och domän.")
            else:
                r.raise_for_status()
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"Request error: {e}. Retry in {wait}s (attempt {attempt}).")
            time.sleep(wait)
    raise RuntimeError("Failed to fetch page after retries")

def main():
    api_key = read_api_key(API_KEY_FILE)
    headers = build_headers(api_key)
    session = requests.Session()

    page = 1
    total = 0
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

    print(f"Fetching tickets updated_since={UPDATED_SINCE} from domain {DOMAIN} ...")
    with OUTPUT_FILE.open("w", encoding="utf-8") as out_f:
        while True:
            data = fetch_page(session, headers, page, PER_PAGE, UPDATED_SINCE)
            if not isinstance(data, list):
                print(f"Unexpected response format on page {page}: {type(data)}")
                break
            if not data:
                print(f"No results on page {page}. Stopping.")
                break
            for ticket in data:
                out_f.write(json.dumps(ticket, ensure_ascii=False) + "\n")
            fetched = len(data)
            total += fetched
            print(f"Page {page}: fetched {fetched} tickets (total {total})")
            if fetched < PER_PAGE:
                break
            page += 1
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"\n✔ Done. Saved {total} tickets to {OUTPUT_FILE.resolve()}")

if __name__ == "__main__":
    main()


Fetching tickets updated_since=2025-05-01 from domain intersolia ...
Page 1: fetched 100 tickets (total 100)
Page 2: fetched 100 tickets (total 200)
Page 3: fetched 100 tickets (total 300)
Page 4: fetched 100 tickets (total 400)
Page 5: fetched 100 tickets (total 500)
Page 6: fetched 100 tickets (total 600)
Page 7: fetched 100 tickets (total 700)
Page 8: fetched 100 tickets (total 800)
Page 9: fetched 100 tickets (total 900)
Page 10: fetched 100 tickets (total 1000)
Page 11: fetched 100 tickets (total 1100)
Page 12: fetched 100 tickets (total 1200)
Page 13: fetched 100 tickets (total 1300)
Page 14: fetched 100 tickets (total 1400)
Page 15: fetched 100 tickets (total 1500)
Page 16: fetched 100 tickets (total 1600)
Page 17: fetched 100 tickets (total 1700)
Page 18: fetched 100 tickets (total 1800)
Page 19: fetched 100 tickets (total 1900)
Page 20: fetched 100 tickets (total 2000)
Page 21: fetched 100 tickets (total 2100)
Page 22: fetched 100 tickets (total 2200)
Page 23: fetched 100 tick

HTTPError: 400 Client Error: Bad Request for url: https://intersolia.freshdesk.com/api/v2/tickets?page=301&per_page=100&updated_since=2025-05-01